In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

# Load dataset
X_train = np.load('/path/to/training/data/X_train.npy')  # Replace with actual path
Y_train = np.load('/path/to/training/data/Y_train.npy')  # Replace with actual path

# Add depth dimension (if needed)
X_train = np.expand_dims(X_train, axis=-1)  # Shape becomes (529, 26, 21, 3, 1)

# Define 3D Fully Convolutional Network (FCN) Model
def build_fcn_3d(input_shape):
    inputs = keras.Input(shape=input_shape)

    # First Conv Block
    x = layers.Conv3D(32, (3, 3, 3), activation='relu', padding='same',
                      kernel_regularizer=regularizers.l2(0.001))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling3D((2, 2, 1))(x)  # Changed from (2,2,2) to (2,2,1)

    # Second Conv Block
    x = layers.Conv3D(64, (3, 3, 3), activation='relu', padding='same',
                      kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling3D((2, 2, 1))(x)  # Prevents depth from becoming 0

    # Third Conv Block
    x = layers.Conv3D(128, (3, 3, 3), activation='relu', padding='same',
                      kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling3D()(x)  # FCN: No Dense layers

    # Output layer for regression
    outputs = layers.Dense(1, activation='linear')(x)

    model = keras.Model(inputs, outputs, name="3D_FCN_Regression")
    return model

# Build and compile model
model = build_fcn_3d(X_train.shape[1:])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss='mean_squared_error',
              metrics=['mae'])

# Print model summary
model.summary()

# Train the model
history = model.fit(X_train, Y_train, epochs=50, batch_size=16, verbose=1)

# Save the trained model
model.save("3D_FCN_model.h5")


In [ ]:
import os
import tensorflow as tf

# Load the saved model
model_path = "3D_FCN_model.h5"  # Change if needed
model = tf.keras.models.load_model(model_path)

# Print model summary
print("\n📌 Model Summary:")
model.summary()

# Get total parameters
total_params = model.count_params()
print(f"\n📊 Total Trainable Parameters: {total_params:,}")

# Print model save directory
absolute_path = os.path.abspath(model_path)
print(f"\n💾 Model saved at: {absolute_path}")


In [ ]:
import numpy as np
import tensorflow as tf

# Load the validation dataset
X_val = np.load("/path/to/validation/data/X_val.npy")  # Shape: (157, 26, 21, 3)
Y_val = np.load("/path/to/validation/data/Y_val.npy")  # Shape: (157,)

# Add a depth dimension to X_val to match the model's expected input shape
X_val = np.expand_dims(X_val, axis=-1)  # New shape: (157, 26, 21, 3, 1)

# Load the trained model
model = tf.keras.models.load_model("/path/to/trained/3D_FCN_model.h5")

# Predict Y values for validation dataset
Y_pred = model.predict(X_val)

# Compute squared error (element-wise)
squared_errors = np.square(Y_pred.flatten() - Y_val)

# Print results
print(f"\n🔹 Number of validation samples: {len(Y_val)}")
print(f"🔹 Squared Errors:\n{squared_errors}")

# Save squared errors to a file (optional)
np.save("squared_errors.npy", squared_errors)

# Print summary statistics
print(f"\n📊 Mean Squared Error: {np.mean(squared_errors):.6f}")
print(f"📉 Min Squared Error: {np.min(squared_errors):.6f}")
print(f"📈 Max Squared Error: {np.max(squared_errors):.6f}")


In [ ]:
import numpy as np
import tensorflow as tf

# Load the training dataset
X_train = np.load("/path/to/training/data/X_train.npy")  # Shape: (529, 26, 21, 3)
Y_train = np.load("/path/to/training/data/Y_train.npy")  # Shape: (529,)

# Add a depth dimension to X_train to match the model's expected input shape
X_train = np.expand_dims(X_train, axis=-1)  # New shape: (529, 26, 21, 3, 1)

# Load the trained model
model = tf.keras.models.load_model("/path/to/trained/3D_FCN_model.h5")

# Predict Y values for training dataset
Y_pred_train = model.predict(X_train)

# Compute squared error (element-wise)
squared_errors_train = np.square(Y_pred_train.flatten() - Y_train)

# Print results
print(f"\n🔹 Number of training samples: {len(Y_train)}")
print(f"🔹 Squared Errors:\n{squared_errors_train}")

# Save squared errors to a file (optional)
np.save("squared_errors_train.npy", squared_errors_train)

# Print summary statistics
print(f"\n📊 Mean Squared Error: {np.mean(squared_errors_train):.6f}")
print(f"📉 Min Squared Error: {np.min(squared_errors_train):.6f}")
print(f"📈 Max Squared Error: {np.max(squared_errors_train):.6f}")


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_squared_error

# Load the testing dataset
X_test = np.load("/path/to/testing/data/X_test.npy")  # Shape: (157, 26, 21, 3)
Y_test = np.load("/path/to/testing/data/Y_test.npy")  # Shape: (157,)

# Add a depth dimension to match the model's expected input shape
X_test = np.expand_dims(X_test, axis=-1)  # New shape: (157, 26, 21, 3, 1)

# Load the trained model
model = tf.keras.models.load_model("/path/to/trained/3D_FCN_model.h5")

# Predict Y values for the test dataset
Y_pred_test = model.predict(X_test)

# Compute RMSE
rmse_test = np.sqrt(mean_squared_error(Y_test, Y_pred_test.flatten()))

# Print results
print(f"\n🔹 Number of testing samples: {len(Y_test)}")
print(f"📊 Root Mean Squared Error (RMSE) on Test Set: {rmse_test:.6f}")


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

# Load the testing dataset
X_test = np.load("/path/to/testing/data/X_test.npy")  # Shape: (157, 26, 21, 3)
Y_test = np.load("/path/to/testing/data/Y_test.npy")  # Shape: (157,)

# Add a depth dimension to match the model's expected input shape
X_test = np.expand_dims(X_test, axis=-1)  # New shape: (157, 26, 21, 3, 1)

# Load the trained model
model = tf.keras.models.load_model("/path/to/trained/3D_FCN_model.h5")

# Predict Y values for the test dataset
Y_pred = model.predict(X_test)

# Compute R² Score
r2 = r2_score(Y_test, Y_pred)

print(f"Coefficient of Determination (R²) on Testing Set: {r2:.4f}")
